# Resource Allocation Log Comparison — Results

Compares two simulated event logs using the evaluation methodology.

**Methodology:** XES logs loaded with pm4py; lifecycle-aware active intervals yield durations from `start|resume → complete|withdraw|ate_abort` (`suspend` is treated as non-working interruption and not counted as active work); non-positive durations discarded. FTE rate κ = 50 €/h. Metrics: cycle time, throughput, handovers, busy hours, C_total = Σ B_r · κ, cost per case, WRF, Gini (G < 0.3 low concentration; 0.3–0.6 moderate; > 0.6 bottleneck risk), termination score S_r with weights α₁=0.25, α₂=0.25, α₃=0.20, α₄=0.15, α₅=0.10, α₆=0.05.

**Logs compared:**
- **Log A:** simulated_log_with_termination
- **Log B:** full_balanced simulated_log

## 1 — Setup & Configuration

In [32]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pm4py

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)

NOTEBOOK_DIR = Path(os.path.abspath("."))
REPO_ROOT = NOTEBOOK_DIR.parent

COST_PER_FTE_HOUR = 50.0  # κ (€/h)

# FTE accounting mode:
# True  -> count only active segments that end with lifecycle='complete'
# False -> count all observed active segments ending with complete/withdraw/ate_abort
FTE_COMPLETE_ONLY = False

LOG_A_PATH = REPO_ROOT / "output" / "test" / "simulated_log_with_termination.xes"
LOG_A_LABEL = "simulated_log_with_termination"

LOG_B_PATH = REPO_ROOT / "integration" / "output" / "full_balanced" / "simulated_log.xes"
LOG_B_LABEL = "full_balanced simulated_log"

if not LOG_A_PATH.exists():
    raise FileNotFoundError(f"Log A not found: {LOG_A_PATH}")
if not LOG_B_PATH.exists():
    raise FileNotFoundError(f"Log B not found: {LOG_B_PATH}")

print(f"Log A: {LOG_A_PATH} ({LOG_A_LABEL})")
print(f"Log B: {LOG_B_PATH} ({LOG_B_LABEL})")

Log A: c:\TUM Master\Business Process Prediction, Simulation and Optimization\Assignment 2\process-simulation-engine\output\test\simulated_log_with_termination.xes (simulated_log_with_termination)
Log B: c:\TUM Master\Business Process Prediction, Simulation and Optimization\Assignment 2\process-simulation-engine\integration\output\full_balanced\simulated_log.xes (full_balanced simulated_log)


## 2 — Load & Validate Both Logs

In [33]:
REQUIRED_COLS = {"case:concept:name", "concept:name", "time:timestamp"}


def load_log(path: Path) -> pd.DataFrame:
    if str(path).endswith(".csv"):
        df = pd.read_csv(path)
    else:
        log = pm4py.read_xes(str(path))
        df = pm4py.convert_to_dataframe(log)
    df.columns = [c.lower() for c in df.columns]
    rename_map = {
        "case:concept:name": "case:concept:name",
        "concept:name": "concept:name",
        "time:timestamp": "time:timestamp",
        "org:resource": "org:resource",
        "lifecycle:transition": "lifecycle:transition",
    }
    df = df.rename(columns={c.lower(): c for c in rename_map})
    missing = REQUIRED_COLS - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], utc=True, format="ISO8601")
    return df


def filter_start_complete(df: pd.DataFrame) -> pd.DataFrame:
    if "lifecycle:transition" not in df.columns:
        return df
    mask = df["lifecycle:transition"].astype(str).str.lower().isin(["start", "complete"])
    return df[mask].copy()


def describe_log(df: pd.DataFrame, label: str) -> None:
    n_cases = df["case:concept:name"].nunique()
    n_events = len(df)
    has_resource = "org:resource" in df.columns
    has_lifecycle = "lifecycle:transition" in df.columns
    print(f"[{label}] Cases: {n_cases:,} | Events: {n_events:,} | Resource: {'✓' if has_resource else '✗'} | Lifecycle: {'✓' if has_lifecycle else '✗'}")


df_a = load_log(LOG_A_PATH)
df_b = load_log(LOG_B_PATH)

describe_log(df_a, LOG_A_LABEL)
describe_log(df_b, LOG_B_LABEL)

[simulated_log_with_termination] Cases: 100 | Events: 4,787 | Resource: ✓ | Lifecycle: ✓
[full_balanced simulated_log] Cases: 100 | Events: 6,883 | Resource: ✓ | Lifecycle: ✓


## Results — Side-by-Side Comparison

## 3 — Metric Computation Functions

In [42]:
def extract_activity_durations(df: pd.DataFrame) -> tuple[pd.DataFrame, bool]:
    has_lifecycle = "lifecycle:transition" in df.columns
    if not has_lifecycle:
        return pd.DataFrame(), False

    df_sorted = df.sort_values(["case:concept:name", "time:timestamp"]).reset_index(drop=True).copy()
    df_sorted["_lc"] = df_sorted["lifecycle:transition"].astype(str).str.lower()

    active_start_transitions = {"start", "resume"}
    pause_transition = "suspend"
    terminal_transitions = {"complete", "withdraw", "ate_abort"}

    merge_cols = ["case:concept:name", "concept:name"]
    if "org:resource" in df.columns:
        merge_cols.append("org:resource")

    rows = []
    for key, sub in df_sorted.groupby(merge_cols, dropna=False):
        sub = sub.sort_values("time:timestamp")

        instance_start_time = None
        active_segment_start = None
        active_seconds = 0.0

        for _, rec in sub.iterrows():
            ts = rec["time:timestamp"]
            lc = rec["_lc"]

            if lc in active_start_transitions:
                if instance_start_time is None:
                    instance_start_time = ts
                if active_segment_start is None:
                    active_segment_start = ts
                continue

            if lc == pause_transition:
                active_segment_start = None
                continue

            if lc in terminal_transitions:
                if active_segment_start is not None and ts > active_segment_start:
                    active_seconds += (ts - active_segment_start).total_seconds()
                active_segment_start = None

                if instance_start_time is not None and active_seconds > 0:
                    row = {}
                    if isinstance(key, tuple):
                        for idx, col in enumerate(merge_cols):
                            row[col] = key[idx]
                    else:
                        row[merge_cols[0]] = key
                    row["start_time"] = instance_start_time
                    row["complete_time"] = ts
                    row["duration_hours"] = active_seconds / 3600
                    row["terminal_transition"] = lc
                    rows.append(row)

                instance_start_time = None
                active_seconds = 0.0
                continue

            # schedule / unknown transitions are ignored

    df_dur = pd.DataFrame(rows)
    if df_dur.empty:
        return pd.DataFrame(), False
    df_dur = df_dur[df_dur["duration_hours"] > 0].reset_index(drop=True)
    return df_dur, not df_dur.empty


def _extract_resource_active_segments(df_raw: pd.DataFrame) -> pd.DataFrame:
    required = {"case:concept:name", "time:timestamp", "org:resource", "lifecycle:transition"}
    if not required.issubset(df_raw.columns):
        return pd.DataFrame()

    df = df_raw.sort_values(["case:concept:name", "time:timestamp"]).copy()
    df["_lc"] = df["lifecycle:transition"].astype(str).str.lower()

    start_transitions = {"start", "resume"}
    terminal_transitions = {"complete", "withdraw", "ate_abort"}

    rows = []
    for key, sub in df.groupby(["case:concept:name", "concept:name"], dropna=False):
        case_id, activity = key
        active_since: dict[str, pd.Timestamp] = {}

        for _, rec in sub.iterrows():
            resource = rec.get("org:resource")
            if pd.isna(resource):
                continue
            resource = str(resource)
            ts = rec["time:timestamp"]
            lc = rec["_lc"]

            if lc in start_transitions:
                if resource not in active_since:
                    active_since[resource] = ts
                continue

            if lc == "suspend" and resource in active_since:
                active_since.pop(resource)
                continue

            if lc in terminal_transitions and resource in active_since:
                start_ts = active_since.pop(resource)
                if ts > start_ts:
                    rows.append({
                        "case:concept:name": case_id,
                        "activity": activity,
                        "resource": resource,
                        "start_time": start_ts,
                        "complete_time": ts,
                        "duration_hours": (ts - start_ts).total_seconds() / 3600.0,
                        "terminal_transition": lc,
                    })

    seg = pd.DataFrame(rows)
    if seg.empty:
        return seg
    return seg[seg["duration_hours"] > 0].reset_index(drop=True)


def _extract_all_working_resources(df_raw: pd.DataFrame) -> pd.DataFrame:
    if "org:resource" not in df_raw.columns:
        return pd.DataFrame(columns=["resource"])

    df_work = df_raw[df_raw["org:resource"].notna()].copy()
    if "lifecycle:transition" in df_work.columns:
        lc = df_work["lifecycle:transition"].astype(str).str.lower()
        work_markers = {"start", "resume", "suspend", "complete", "withdraw", "ate_abort"}
        df_work = df_work[lc.isin(work_markers)]

    resources = (
        df_work["org:resource"]
        .astype(str)
        .dropna()
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )
    return resources.to_frame(name="resource")


def compute_case_metrics(df: pd.DataFrame) -> pd.DataFrame:
    grp = df.sort_values("time:timestamp").groupby("case:concept:name")
    case_start = grp["time:timestamp"].min().rename("case_start")
    case_end = grp["time:timestamp"].max().rename("case_end")
    n_events = grp.size().rename("n_events")
    df_cases = pd.concat([case_start, case_end, n_events], axis=1)
    df_cases["cycle_time_days"] = (df_cases["case_end"] - df_cases["case_start"]).dt.total_seconds() / 86_400
    if "org:resource" in df.columns:
        def count_handovers(sub):
            r = sub.sort_values("time:timestamp")["org:resource"].dropna().tolist()
            return sum(1 for a, b in zip(r, r[1:]) if a != b)
        df_cases["handovers"] = grp.apply(count_handovers)
    else:
        df_cases["handovers"] = np.nan
    return df_cases.reset_index()


def compute_resource_metrics(df_durations: pd.DataFrame, has_durations: bool) -> pd.DataFrame | None:
    if "lifecycle:transition" not in df_a.columns and "lifecycle:transition" not in df_b.columns:
        return None

    # use df_durations source ownership via available columns; fallback is safe here
    source_df = df_a if df_durations is not None and len(df_durations) >= 0 else df_a
    _ = source_df  # keep explicit for readability

    # compute from global df_raw passed in compute_all_metrics; this function is retained for structure
    if not has_durations:
        return None
    return None


def compute_all_metrics(df_raw: pd.DataFrame, cost_per_fte: float) -> dict:
    df_dur, has_dur = extract_activity_durations(df_raw)
    df_cases = compute_case_metrics(df_raw)
    all_resources = _extract_all_working_resources(df_raw)

    df_busy_all = _extract_resource_active_segments(df_raw)
    if all_resources.empty:
        df_res = None
        total_busy = np.nan
        fte_cost = np.nan
    else:
        if df_busy_all.empty:
            df_busy_agg = pd.DataFrame(columns=["resource", "busy_hours"])
        else:
            df_busy_agg = (
                df_busy_all.groupby("resource")["duration_hours"]
                .sum()
                .reset_index()
                .rename(columns={"duration_hours": "busy_hours"})
            )

        df_res = (
            all_resources.merge(df_busy_agg, on="resource", how="left")
            .fillna({"busy_hours": 0.0})
            .sort_values("busy_hours", ascending=False)
            .reset_index(drop=True)
        )
        total_busy = float(df_res["busy_hours"].sum())
        fte_cost = total_busy * cost_per_fte

        total_share = df_res["busy_hours"].sum()
        df_res["work_share"] = df_res["busy_hours"] / total_share if total_share > 0 else 0.0

    n = len(df_cases)

    def safe_mean(s):
        return float(s.mean()) if s.notna().any() else np.nan

    rm = compute_resource_features(df_raw, df_dur, df_cases, has_dur, cost_per_fte)
    max_term, mean_term = np.nan, np.nan
    if rm is not None and len(rm) > 0:
        for col, norm in [("fte_cost", "norm_cost"), ("productivity", "norm_productivity"), ("contribution_per_cost", "norm_value"),
                          ("distinct_cases_count", "norm_contribution"), ("workload_share", "norm_workload_share"), ("cycle_time_delta", "norm_cycle_delta")]:
            rm[norm] = percentile_normalize(rm[col])
        rm["termination_score"] = (
            W_COST * rm["norm_cost"]
            + W_LOW_PRODUCTIVITY * (1 - rm["norm_productivity"] )
            + W_LOW_VALUE * (1 - rm["norm_value"] )
            + W_LOW_CONTRIBUTION * (1 - rm["norm_contribution"] )
            + W_CRITICALITY * (1 - rm["norm_workload_share"] )
            + W_CYCLE_IMPACT * rm["norm_cycle_delta"]
        )
        max_term = float(rm["termination_score"].max())
        mean_term = float(rm["termination_score"].mean())

    return {
        "Avg Cycle Time (days)": float(df_cases["cycle_time_days"].mean()),
        "Median Cycle Time (days)": float(df_cases["cycle_time_days"].median()),
        "Range Cycle Time (days)": float(df_cases["cycle_time_days"].max() - df_cases["cycle_time_days"].min()),
        "Throughput (cases/day)": n / ((df_cases["case_end"].max() - df_cases["case_start"].min()).total_seconds() / 86400) if n > 0 else np.nan,
        "Avg Resource Occupation (share)": float(df_res["work_share"].mean()) if df_res is not None else np.nan,
        "Weighted Resource Fairness": _wrf(df_res) if df_res is not None else np.nan,
        "Workload Gini Coefficient": gini_coefficient(df_res["busy_hours"].values) if df_res is not None else np.nan,
        "Total Busy Hours": total_busy,
        "Total FTE Cost (€)": fte_cost,
        "Cost per Completed Case (€)": fte_cost / n if n > 0 and not np.isnan(fte_cost) else np.nan,
        "Avg Handovers per Case": safe_mean(df_cases["handovers"]),
        "Number of Resources": len(df_res) if df_res is not None else np.nan,
        "Max Termination Score": max_term,
        "Mean Termination Score": mean_term,
    }


def _wrf(df_res: pd.DataFrame) -> float:
    shares = df_res["work_share"].values
    weights = df_res["busy_hours"].values / df_res["busy_hours"].sum()
    return float(np.sum(weights * np.abs(shares - shares.mean())))


def gini_coefficient(values: np.ndarray) -> float:
    v = np.sort(np.asarray(values, dtype=float))
    v = v[v >= 0]
    n = len(v)
    if n == 0 or v.sum() == 0:
        return np.nan
    ranks = np.arange(1, n + 1)
    return float((2 * (ranks * v).sum()) / (n * v.sum()) - (n + 1) / n)


def compute_resource_features(df_raw, df_durations, df_cases, has_durations, cost_per_fte):
    if "org:resource" not in df_raw.columns or "lifecycle:transition" not in df_raw.columns:
        return None

    df_busy_all = _extract_resource_active_segments(df_raw)
    all_resources = _extract_all_working_resources(df_raw)
    if all_resources.empty:
        return None

    if df_busy_all.empty:
        busy_base = pd.DataFrame(columns=["resource", "busy_hours"])
    else:
        busy_base = (
            df_busy_all.groupby("resource")["duration_hours"]
            .sum()
            .reset_index()
            .rename(columns={"duration_hours": "busy_hours"})
        )

    busy = all_resources.merge(busy_base, on="resource", how="left").fillna({"busy_hours": 0.0})
    busy["fte_cost"] = busy["busy_hours"] * cost_per_fte
    total_busy = busy["busy_hours"].sum()
    busy["workload_share"] = busy["busy_hours"] / total_busy if total_busy > 0 else 0.0

    df_lc = df_raw.copy()
    df_lc["_lc"] = df_lc["lifecycle:transition"].astype(str).str.lower()

    complete_events = df_lc[df_lc["_lc"] == "complete"].copy()
    negative_events = df_lc[df_lc["_lc"].isin(["withdraw", "ate_abort"])].copy()

    event_counts = (
        complete_events.groupby("org:resource")
        .size()
        .reset_index(name="completed_events_count")
        .rename(columns={"org:resource": "resource"})
    )
    case_counts = (
        complete_events.groupby("org:resource")["case:concept:name"]
        .nunique()
        .reset_index()
        .rename(columns={"org:resource": "resource", "case:concept:name": "distinct_cases_count"})
    )
    negative_counts = (
        negative_events.groupby("org:resource")
        .size()
        .reset_index(name="negative_outcomes_count")
        .rename(columns={"org:resource": "resource"})
    )

    case_ct = df_cases[["case:concept:name", "cycle_time_days"]].copy()
    resource_cases = df_busy_all[["resource", "case:concept:name"]].drop_duplicates()
    resource_ct = (
        resource_cases.merge(case_ct, on="case:concept:name", how="left")
        .groupby("resource")["cycle_time_days"]
        .mean()
        .reset_index()
        .rename(columns={"cycle_time_days": "avg_cycle_time_of_cases_involved"})
    )

    global_avg_ct = df_cases["cycle_time_days"].mean()
    rm = (
        busy.merge(event_counts, on="resource", how="left")
        .merge(case_counts, on="resource", how="left")
        .merge(negative_counts, on="resource", how="left")
        .merge(resource_ct, on="resource", how="left")
    )
    for c in ["completed_events_count", "distinct_cases_count", "negative_outcomes_count"]:
        rm[c] = rm[c].fillna(0.0)

    rm["productivity"] = np.where(rm["busy_hours"] > 0, rm["completed_events_count"] / rm["busy_hours"], 0.0)
    rm["contribution_per_cost"] = np.where(rm["fte_cost"] > 0, rm["distinct_cases_count"] / rm["fte_cost"], 0.0)
    rm["completion_success_rate"] = np.where(
        (rm["completed_events_count"] + rm["negative_outcomes_count"]) > 0,
        rm["completed_events_count"] / (rm["completed_events_count"] + rm["negative_outcomes_count"]),
        np.nan,
    )
    rm["cycle_time_delta"] = rm["avg_cycle_time_of_cases_involved"] - global_avg_ct
    return rm


def percentile_normalize(series: pd.Series, p_low: float = 5, p_high: float = 95) -> pd.Series:
    lo, hi = np.percentile(series.dropna(), p_low), np.percentile(series.dropna(), p_high)
    clipped = series.clip(lower=lo, upper=hi)
    return pd.Series(0.5, index=series.index) if (hi - lo) == 0 else (clipped - lo) / (hi - lo)


# Termination score weights: α₁=0.25, α₂=0.25, α₃=0.20, α₄=0.15, α₅=0.10, α₆=0.05
W_COST, W_LOW_PRODUCTIVITY, W_LOW_VALUE = 0.25, 0.25, 0.20
W_LOW_CONTRIBUTION, W_CRITICALITY, W_CYCLE_IMPACT = 0.15, 0.10, 0.05

## 4 — Side-by-Side Comparison

In [43]:
metrics_a = compute_all_metrics(df_a, COST_PER_FTE_HOUR)
metrics_b = compute_all_metrics(df_b, COST_PER_FTE_HOUR)

comparison = pd.DataFrame({
    LOG_A_LABEL: metrics_a,
    LOG_B_LABEL: metrics_b,
})
comparison["Difference (B - A)"] = comparison[LOG_B_LABEL] - comparison[LOG_A_LABEL]

def pct_change(a, b):
    if np.isnan(a) or np.isnan(b) or a == 0:
        return np.nan
    return (b - a) / a * 100

comparison["% Change"] = [pct_change(metrics_a[k], metrics_b[k]) for k in comparison.index]

def fte_breakdown(df_raw: pd.DataFrame, label: str) -> pd.DataFrame:
    seg = _extract_resource_active_segments(df_raw)
    if seg.empty:
        return pd.DataFrame({"Log": [label], "Info": ["No lifecycle segments found"]})

    breakdown = (
        seg.groupby("terminal_transition", dropna=False)["duration_hours"]
        .sum()
        .reset_index()
        .rename(columns={"duration_hours": "busy_hours"})
    )
    breakdown["fte_cost_eur"] = breakdown["busy_hours"] * COST_PER_FTE_HOUR
    breakdown["Log"] = label
    return breakdown[["Log", "terminal_transition", "busy_hours", "fte_cost_eur"]].sort_values("busy_hours", ascending=False)

breakdown_a = fte_breakdown(df_a, LOG_A_LABEL)
breakdown_b = fte_breakdown(df_b, LOG_B_LABEL)

print("FTE mode (both logs): active deltas counted only for start/resume -> complete/withdraw/ate_abort (suspend excluded)")
display(comparison.round(4))
display(pd.concat([breakdown_a, breakdown_b], ignore_index=True).round(4))

FTE mode (both logs): active deltas counted only for start/resume -> complete/withdraw/ate_abort (suspend excluded)


,simulated_log_with_termination,full_balanced simulated_log,Difference (B - A),% Change
Avg Cycle Time (days),34.7884,44.4204,9.6319,27.6871
Median Cycle Time (days),25.0474,20.2012,-4.8462,-19.3482
Range Cycle Time (days),292.1256,358.1629,66.0373,22.6058
Throughput (cases/day),0.3351,0.2697,-0.0654,-19.5200
Avg Resource Occupation (share),0.0118,0.0076,-0.0041,-35.1145
Weighted Resource Fairness,0.2892,0.9773,0.6881,237.9100
Workload Gini Coefficient,0.9639,0.9923,0.0283,2.9375
Total Busy Hours,626.6324,12249.7192,11623.0867,1854.8492
Total FTE Cost (€),31331.6218,612485.9587,581154.3369,1854.8492
Cost per Completed Case (€),313.3162,6124.8596,5811.5434,1854.8492


,Log,terminal_transition,busy_hours,fte_cost_eur
0,simulated_log_with_termination,ate_abort,451.8035,22590.1753
1,simulated_log_with_termination,complete,174.8289,8741.4465
2,full_balanced simulated_log,complete,10034.9049,501745.2472
3,full_balanced simulated_log,ate_abort,2214.8142,110740.7114


In [ ]:
def _count_working_resources(df_raw: pd.DataFrame) -> int:
    if "org:resource" not in df_raw.columns:
        return 0
    df_work = df_raw[df_raw["org:resource"].notna()].copy()
    if "lifecycle:transition" in df_work.columns:
        lc = df_work["lifecycle:transition"].astype(str).str.lower()
        work_markers = {"start", "resume", "suspend", "complete", "withdraw", "ate_abort"}
        df_work = df_work[lc.isin(work_markers)]
    return int(df_work["org:resource"].astype(str).nunique())

print("Working resources in logs (event-based):")
print(f"- {LOG_A_LABEL}: {_count_working_resources(df_a)}")
print(f"- {LOG_B_LABEL}: {_count_working_resources(df_b)}")
print("\nResources used in metrics (after merge with busy hours):")
print(f"- {LOG_A_LABEL}: {metrics_a['Number of Resources']}")
print(f"- {LOG_B_LABEL}: {metrics_b['Number of Resources']}")